In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
%matplotlib inline

import inversion_tools

#start_date = '2016-01-01'
#end_date   = '2016-06-01'
start_date = '2014-09-01'
end_date   = '2019-01-01'
pdir  = '/work/noaa/co2/jhollo/processed_transport_data/'

In [ ]:
# ------------ read residual emissions data for ocean from region 14, gpp and resp from region 3
args  = {'start_date':start_date, 'end_date':end_date, 'month':'2016-01', 'processing_dir':pdir, 'return_flux':False, 'quiet':True}
ocean_mf_r14 = inversion_tools.read_transport_jacobians_residual('ocean', region=14, **args)
resp_mf_r3   = inversion_tools.read_transport_jacobians_residual('resp', region=3, pft=4, **args)
gpp_mf_r3    = inversion_tools.read_transport_jacobians_residual('gpp', region=3, pft=4, **args)

reading data for splits [ 1  2  3  4  5  6  7  8  9 10 11 12 13]
---------- split 1 ----------
reading mole fraction data for all vertical levels...
took 7.77 s
done
---------- split 2 ----------
reading mole fraction data for all vertical levels...
took 7.42 s
done
---------- split 3 ----------
reading mole fraction data for all vertical levels...
took 8.37 s
done
---------- split 4 ----------
reading mole fraction data for all vertical levels...
took 7.35 s
done
---------- split 5 ----------
reading mole fraction data for all vertical levels...
took 7.50 s
done
---------- split 6 ----------
reading mole fraction data for all vertical levels...


In [ ]:
# ------------ read control emissions data for ocean, gpp, resp
args  = {'start_date':start_date, 'end_date':end_date, 'processing_dir':pdir, 'return_mf':False, 'quiet':True}
ocean_flx = inversion_tools.read_transport_jacobians_control('ocean', **args)
resp_flx  = inversion_tools.read_transport_jacobians_control('resp', pft=4, **args)
gpp_flx   = inversion_tools.read_transport_jacobians_control('gpp', pft=4, **args)

In [ ]:
# ------------ get time series averaged on the Nino-3.4 region
weighted_lat_avg = lambda data: data.weighted(np.cos(np.deg2rad(data.lat))).mean('lat')
lon_avg          = lambda data: data.mean('lon')
horz_avg         = lambda data: lon_avg(weighted_lat_avg(data))
nino_region      = {'lat':slice(-5, 5), 'lon':slice(-170, -120)}
amazon_region    = {'lat':slice(-15, 15), 'lon':slice(-80, -45)}

ocean_flx_nino   = horz_avg(ocean_flx.sel(nino_region))
resp_flx_amazon  = horz_avg(resp_flx.sel(amazon_region))
gpp_flx_amazon = horz_avg(ocean_flx.sel(amazon_region))

In [ ]:
# ------------ plot time series for fluxes
net_flux = lambda data: data['intercept'] + data['trend'] + data['sin1'] + data['sin2'] + data['cos1'] + data['cos2'] + data['residual']

def plot_flux_timeseries(x, y, r, label, title, color='r', twinx=False, yfrac=0):
    fig = plt.figure(figsize=(9,4))
    ax  = fig.add_subplot()
    if(twinx): ax2 = ax.twinx()
    else:      ax2 = ax
    ax.plot(x, r, '-', color=color, label='residual')
    ax2.plot(x, y-r, '-k', label='intercept+trend+seasonal')
    if(twinx):
        ax.plot(x[0], r[0], '-k', label='intercept+trend+seasonal')
    ylim = ax.get_ylim()
    ypad = abs(ylim[0]-ylim[1]) * yfrac
    ylim2 = ax2.get_ylim()
    ypad2 = abs(ylim[0]-ylim[1]) * yfrac
    ax.set_ylim([ylim[0] - ypad, ylim[1] + ypad])
    ax2.set_ylim([ylim2[0] - ypad2, ylim2[1] + ypad2])
    ax.set_ylabel(f'{label} flux [g m2/day]')
    ax2.set_ylabel(f'{label} flux [g m2/day]')
    ax.legend()
    ax.set_title(title)
    ax.axhline(y=0, ls='--', color='k', lw=1)

# ---- ocean
y = net_flux(ocean_flx_nino)
r = ocean_flx_nino['residual']
x = y.time
plot_flux_timeseries(x, y, r, 'ocean', 'Weighted mean over lat=(-5, 5), lon=(-170, -120)', 'b')

# ---- resp
y = net_flux(resp_flx_amazon)
r = resp_flx_amazon['residual']
x = y.time
plot_flux_timeseries(x, y, r, 'resp', 'Weighted mean over lat=(-15, 15), lon=(-80, -45)', 'r', True, 0.2)

# ---- gpp
y = net_flux(gpp_flx_amazon)
r = gpp_flx_amazon['residual']
x = y.time
plot_flux_timeseries(x, y, r, 'gpp', 'Weighted mean over lat=(-15, 15), lon=(-80, -45)', 'g')

# ---- resp + gpp
y = net_flux(gpp_flx_amazon) + net_flux(resp_flx_amazon)
r = gpp_flx_amazon['residual'] + resp_flx_amazon['residual']
x = y.time
plot_flux_timeseries(x, y, r, 'gpp+resp', 'Weighted mean over lat=(-15, 15), lon=(-80, -45)', 'g', True, 0.2)


In [ ]:
# ------------ plot zonal-mean time series for mf at select pressure levels

# ---- ocean
dd = ocean_mf_r14['residual'].mean('lon').isel(lev=35).isel(time=slice(0, 900))
levels = [-9, -8, -7, -6, -5, -4, -3, -2, -1, 0]
fig = plt.figure()
ax = fig.add_subplot()
cf = ax.contourf(dd.lat, dd.time, dd*1e3, levels=levels, extend='both', cmap='YlGnBu_r')
ax.contour(dd.lat, dd.time, dd*1e3, levels=levels, colors='k', linestyles='-', linewidths=0.2)
ax.set_xlabel('latitude')
cb = plt.colorbar(cf, location='top')
cb.set_label('100 hPa mole fraction response to Region 14 ocean residual flux [ppb]')

 # ----- resp
dd = resp_mf_r3['residual'].mean('lon').isel(lev=35).isel(time=slice(0, 900))
dd = dd + resp_mf_r3['residual'].mean('lon').isel(lev=35).isel(time=slice(0, 900))
levels = [-9, -8, -7, -6, -5, -4, -3, -2, -1, 0]
fig = plt.figure()
ax = fig.add_subplot()
cf = ax.contourf(dd.lat, dd.time, dd, levels=10, extend='both', cmap='PuBuGn_r')
ax.contour(dd.lat, dd.time, dd, levels=10, colors='k', linestyles='-', linewidths=0.2)
ax.set_xlabel('latitude')
cb = plt.colorbar(cf, location='top')
cb.set_label('100 hPa mole fraction response to Region 3 resp+gpp residual flux [ppm]')

In [ ]:
# ------------ plot zonal-mean time series for mf at select pressure levels

# ---- ocean
dd = ocean_mf_r14['residual'].mean('lon').isel(lev=25).isel(time=slice(0, 900))
levels = [-9, -8, -7, -6, -5, -4, -3, -2, -1, 0]
fig = plt.figure()
ax = fig.add_subplot()
cf = ax.contourf(dd.lat, dd.time, dd*1e3, levels=levels, extend='both', cmap='YlGnBu_r')
ax.contour(dd.lat, dd.time, dd*1e3, levels=levels, colors='k', linestyles='-', linewidths=0.2)
ax.set_xlabel('latitude')
cb = plt.colorbar(cf, location='top')
cb.set_label('430 hPa mole fraction response to Region 14 ocean residual flux [ppb]')

 # ----- resp
dd = resp_mf_r3['residual'].mean('lon').isel(lev=25).isel(time=slice(0, 900))
dd = dd + resp_mf_r3['residual'].mean('lon').isel(lev=25).isel(time=slice(0, 900))
levels = np.array([-9, -8, -7, -6, -5, -4, -3, -2, -1, 0])/10
fig = plt.figure()
ax = fig.add_subplot()
cf = ax.contourf(dd.lat, dd.time, dd, levels=10, extend='both', cmap='PuBuGn_r')
ax.contour(dd.lat, dd.time, dd, levels=10, colors='k', linestyles='-', linewidths=0.2)
ax.set_xlabel('latitude')
cb = plt.colorbar(cf, location='top')
cb.set_label('430 hPa mole fraction response to Region 3 resp+gpp residual flux [ppm]')

In [ ]:
# ------------ plot zonal-mean time series for mf at select pressure levels, and select latitudes

for lev in [38, 35, 25, 15][::-1]:

    pp = inversion_tools.lev_to_p(lev)
    
    # ---- ocean
    dd = ocean_mf_r14['residual'].isel(lev=lev).isel(time=slice(0, 600))
    dd_trop = horz_avg(dd.sel(lat=slice(-20, 20)))
    dd_midlat = horz_avg(dd.sel(lat=slice(30, 50)))
    dd_hilat  = horz_avg(dd.sel(lat=slice(60, 90)))
    
    fig = plt.figure()
    ax = fig.add_subplot()
    ax.plot(dd.time, dd_trop*1e3, '-b', label='lat=[-20, 20]')
    ax.plot(dd.time, dd_midlat*1e3, '--b', label='lat=[30, 50]')
    ax.plot(dd.time, dd_hilat*1e3, '-.b', label='lat=[60, 90]')
    ax.set_xlabel('time')
    ax.set_ylabel('ppb')
    ax.set_title(f'{pp} hPa response to Region 14 ocean residual flux')
    ax.legend()
    for label in ax.get_xmajorticklabels() + ax.get_xmajorticklabels():
        label.set_rotation(30)
        label.set_horizontalalignment("right")
    
     # ----- resp
    dd  = resp_mf_r3['residual'].isel(lev=lev).isel(time=slice(0, 600))
    dd += gpp_mf_r3['residual'].isel(lev=lev).isel(time=slice(0, 600))
    dd_trop   = horz_avg(dd.sel(lat=slice(-20, 20)))
    dd_midlat = horz_avg(dd.sel(lat=slice(30, 50)))
    dd_hilat  = horz_avg(dd.sel(lat=slice(60, 90)))
    
    fig = plt.figure()
    ax = fig.add_subplot()
    ax.plot(dd.time, dd_trop, '-g', label='lat=[-20, 20]')
    ax.plot(dd.time, dd_midlat, '--g', label='lat=[30, 50]')
    ax.plot(dd.time, dd_hilat, '-.g', label='lat=[60, 90]')
    ax.set_xlabel('time')
    ax.set_ylabel('ppm')
    ax.set_title(f'{pp} hPa response to Region 3 resp+gpp residual flux')
    ax.legend()
    for label in ax.get_xmajorticklabels() + ax.get_xmajorticklabels():
        label.set_rotation(30)
        label.set_horizontalalignment("right")

In [ ]:
# ------------ get column-integrated tracers
ocean_xco2_r14 = inversion_tools.column_average(ocean_mf_r14['residual'])
resp_xco2_r3   = inversion_tools.column_average(resp_mf_r3['residual'])
gpp_xco2_r3    = inversion_tools.column_average(gpp_mf_r3['residual'])